# MAL Omics — Proteomic Correlates of Mechanically Affected Lung
**Project:** COPDGene Multi-omics | **Analysis:** Proteomics Aim 2

## Goals
1. **Biological discovery:** identify peripheral plasma proteins associated with MAL (Mechanically Affected Lung)
2. **Biomarker development:** build a parsimonious proteomic risk score to enrich studies for MAL and emphysema progressors

## Analysis Roadmap
| Step | Analysis | Output |
|------|----------|--------|
| 1 | Demographics | Table 1 |
| 2 | Differential protein expression (protein ~ MAL) | Volcano plot, DE table |
| 3 | Gene set enrichment analysis (GSEA) | Pathway barplot |
| 4 | STRING protein–protein interaction network | Network figure |
| 5 | Adaptive LASSO → MAL proteomic risk score | Score weights table |
| 6 | Risk score ~ emphysema progression + AIC comparison | Quartile plot |
| 7 | Causal mediation: MAL → Risk Score → Progression | Table 2 |

## Section 0 — Background

**MAL (Mechanically Affected Lung):** CT-derived continuous score (0.01–0.40) quantifying
the proportion of lung volume undergoing destructive mechanical stress from pressure gradients.
Higher MAL = greater mechanical injury to alveolar tissue.

**Proteomics:** SomaScan 5K platform measures ~4,979 plasma proteins as RFU values (aptamer-based).
Log₂-transformed before analysis: 1-unit change = doubling of protein level.

**Emphysema progression:** `Change_lung_density_vnb_P2_P3` = Phase 3 − Phase 2 CT density (HU).
Negative = lung became less dense = emphysema worsened.

**Statistical rationale for adaptive lasso risk score:**
With n ≈ 1,300 subjects and p ≈ 4,979 proteins (p >> n), ordinary regression is
underdetermined. Adaptive lasso (Zou 2006) selects a parsimonious set of proteins
with oracle variable-selection consistency, then OLS refit yields unbiased effect estimates.

In [ ]:
# =============================================================================
# SECTION 1: PACKAGES AND FILE PATHS
# =============================================================================

suppressPackageStartupMessages({
  # Data wrangling
  library(dplyr); library(tibble); library(readr); library(tidyr); library(purrr)
  # Modelling
  library(glmnet)   # adaptive lasso (ridge + L1 penalized regression)
  library(broom)    # tidy() for lm output
  # Table 1
  library(gtsummary)
  # Visualization
  library(ggplot2); library(ggrepel)
  # Pathway analysis
  library(fgsea); library(msigdbr)
  # STRING network
  library(STRINGdb)
  # Causal mediation
  library(medflex)
})

# File paths — all relative to project root (COPD/Journal/)
BASE       <- here::here()
PROT_PATH  <- file.path(BASE, "data/csv_exports/prot.csv")
PHENO_PATH <- file.path(BASE, "data/csv_exports/pheno_MAL.csv")
MAL_PATH   <- file.path(BASE, "data/Data/MAL.csv")
META_PATH  <- file.path(BASE, "data/csv_exports/metadat5k.csv")
OUT_DIR    <- file.path(BASE, "MAL_omics")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

## Section 2 — Helper Functions

In [ ]:
# -------------------------------------------------------------------
# load_data()
# 3-way inner join of proteins, phenotypes, and MAL.
# Log2-transforms proteins; collapses rare scanner levels;
# defines binary progression outcome.
# Returns: list($df, $protein_cols)
# -------------------------------------------------------------------
load_data <- function(prot_path, pheno_path, mal_path) {
  prot  <- read_csv(prot_path,  show_col_types = FALSE)
  pheno <- read_csv(pheno_path, show_col_types = FALSE)
  mal   <- read_csv(mal_path,   show_col_types = FALSE) |> select(sid, meanmal)

  df <- prot |> inner_join(pheno, by = "sid") |> inner_join(mal, by = "sid")
  message("Rows after 3-way merge: ", nrow(df))

  protein_cols <- grep("^X[0-9]", names(df), value = TRUE)
  message("Proteins: ", length(protein_cols))

  # Log2 transform: RFU values are log-normal; log2 normalises distribution
  df <- df |> mutate(across(all_of(protein_cols), ~ log2(pmax(., 1e-6))))
  protein_cols_log2 <- paste0("log2_", protein_cols)
  names(df)[names(df) %in% protein_cols] <- protein_cols_log2

  # Collapse rare scanner levels (<15 subjects) to prevent factor-level issues
  scanner_counts <- table(df$scanner_model_clean_P2)
  rare_scanners  <- names(scanner_counts[scanner_counts < 15])
  df <- df |> mutate(scanner_model_clean_P2 =
    if_else(scanner_model_clean_P2 %in% rare_scanners, "Other", scanner_model_clean_P2))

  # Binary progression for AUC (used in downstream association checks)
  df <- df |> mutate(progression_binary = as.integer(Change_lung_density_vnb_P2_P3 < 0))

  list(df = df, protein_cols = protein_cols_log2)
}


# -------------------------------------------------------------------
# run_protein_lm()
# Runs lm(protein ~ exposure + covariates) for every protein.
# Extracts the exposure coefficient (β, SE, t, p) and applies BH FDR.
# Returns: data frame with one row per protein
# -------------------------------------------------------------------
run_protein_lm <- function(df, protein_cols, exposure, covariate_formula) {
  results <- vector("list", length(protein_cols))

  for (i in seq_along(protein_cols)) {
    p   <- protein_cols[i]
    fml <- as.formula(paste(p, "~", exposure, "+", covariate_formula))
    mod <- tryCatch(lm(fml, data = df), error = function(e) NULL)
    if (is.null(mod)) next

    s <- summary(mod)$coefficients
    if (!exposure %in% rownames(s)) next

    results[[i]] <- data.frame(
      predictor = p,
      beta      = s[exposure, "Estimate"],
      se        = s[exposure, "Std. Error"],
      t_stat    = s[exposure, "t value"],
      p_value   = s[exposure, "Pr(>|t|)"],
      stringsAsFactors = FALSE
    )
  }

  out <- bind_rows(results) |> as_tibble()
  out$FDR <- p.adjust(out$p_value, method = "BH")
  out
}

In [ ]:
# -------------------------------------------------------------------
# Adaptive lasso helpers (reused from Aim2_prediction notebook)
# build_design / compute_adaptive_weights / fit_adaptive_lasso / refit_ols
# -------------------------------------------------------------------

build_design <- function(df, protein_cols, fixed_vars,
                         adaptive_weights = NULL, ref_colnames = NULL) {
  fixed_formula <- as.formula(paste("~", paste(fixed_vars, collapse = " + "), "- 1"))
  X_fixed <- model.matrix(fixed_formula, data = df)
  n_fixed  <- ncol(X_fixed)
  X_prot   <- as.matrix(df[, protein_cols, drop = FALSE])
  X        <- cbind(X_fixed, X_prot)

  # Align columns to training reference (prevents dummy-column mismatch in splits)
  if (!is.null(ref_colnames)) {
    missing_cols <- setdiff(ref_colnames, colnames(X))
    if (length(missing_cols) > 0) {
      zero_mat <- matrix(0, nrow = nrow(X), ncol = length(missing_cols),
                         dimnames = list(NULL, missing_cols))
      X <- cbind(X, zero_mat)
    }
    X <- X[, ref_colnames, drop = FALSE]
  }

  pf <- c(rep(0, n_fixed),
          if (is.null(adaptive_weights)) rep(1, length(protein_cols)) else adaptive_weights)
  list(X = X, pf = pf, n_fixed = n_fixed, colnames = colnames(X))
}

compute_adaptive_weights <- function(X_prot, y_resid) {
  # Ridge (alpha=0): stable initial estimates when p >> n
  ridge_cv   <- cv.glmnet(x = X_prot, y = y_resid, alpha = 0,
                           nfolds = 10, standardize = TRUE)
  ridge_coef <- as.vector(coef(ridge_cv, s = "lambda.min"))[-1]
  # Adaptive weights: weak predictors get large penalty → zeroed out
  1 / pmax(abs(ridge_coef), 1e-10)
}

fit_adaptive_lasso <- function(X, y, penalty_factor, nfolds = 10) {
  cv.glmnet(x = X, y = y, alpha = 1,
            penalty.factor = penalty_factor,
            nfolds = nfolds, standardize = TRUE)
}

refit_ols <- function(X, y, lasso_model, lambda, protein_cols, n_fixed) {
  lasso_coef       <- as.vector(coef(lasso_model, s = lambda))[-1]
  protein_indices  <- seq(n_fixed + 1, length(lasso_coef))
  selected_prots   <- protein_indices[lasso_coef[protein_indices] != 0]
  selected_all     <- c(seq_len(n_fixed), selected_prots)
  message("Proteins selected: ", length(selected_prots))

  X_sel       <- X[, selected_all, drop = FALSE]
  ols_fit     <- lm(y ~ X_sel)
  ols_summary <- summary(ols_fit)$coefficients
  ols_ci      <- confint(ols_fit, level = 0.95)
  clean_names <- sub("^X_sel", "", rownames(ols_summary))

  tibble(
    predictor = clean_names,
    beta      = ols_summary[, "Estimate"],
    se        = ols_summary[, "Std. Error"],
    ci_lower  = ols_ci[rownames(ols_summary), 1],
    ci_upper  = ols_ci[rownames(ols_summary), 2],
    p_value   = ols_summary[, "Pr(>|t|)"]
  )
}


# -------------------------------------------------------------------
# annotate_proteins()
# Joins OLS results to metadata; ranks by |β|.
# -------------------------------------------------------------------
annotate_proteins <- function(ols_result, protein_cols, meta) {
  ols_result |>
    filter(predictor %in% protein_cols) |>
    mutate(seqID = sub("^log2_X", "", predictor), abs_beta = abs(beta)) |>
    left_join(meta, by = "seqID") |>
    arrange(desc(abs_beta)) |>
    select(gene, target, beta, se, ci_lower, ci_upper, p_value, predictor, seqID)
}

## Section 3 — Load and Preprocess Data

In [ ]:
# Load metadata for protein annotation
meta <- read_csv(META_PATH, show_col_types = FALSE) |>
  select(seqID, gene = EntrezGeneSymbol, target = Target, full_name = TargetFullName)

# Load and preprocess all data
data_obj     <- load_data(PROT_PATH, PHENO_PATH, MAL_PATH)
df_raw       <- data_obj$df
protein_cols <- data_obj$protein_cols

# Identify BMI column (name varies across COPDGene releases)
bmi_col <- grep("^[Bb][Mm][Ii]", names(df_raw), value = TRUE)[1]
if (is.na(bmi_col)) {
  message("WARNING: no BMI column found — omitting from Table 1 and models")
  bmi_col <- NULL
} else {
  message("BMI column: ", bmi_col)
}

# Complete-case filter on all analysis variables
required_cols <- c(
  "Change_lung_density_vnb_P2_P3", "lung_density_vnb_P2", "meanmal",
  "Age_P2", "gender", "race", "ATS_PackYears_P2", "SmokCigNow_P2",
  "scanner_model_clean_P2", "FEV1pp_P2", "FEV1_FVC_post_P2",
  paste0("PC", 1:5)
)
if (!is.null(bmi_col)) required_cols <- c(required_cols, bmi_col)

df_full <- df_raw |> filter(if_all(all_of(required_cols), ~ !is.na(.)))
message("Subjects after complete-case filter: ", nrow(df_full))

## Table 1 — Demographics

In [ ]:
demo_vars <- c("Age_P2", "gender", "race", "SmokCigNow_P2",
               "ATS_PackYears_P2", "FEV1pp_P2", "FEV1_FVC_post_P2",
               "lung_density_vnb_P2", "meanmal", "Change_lung_density_vnb_P2_P3")
if (!is.null(bmi_col)) demo_vars <- append(demo_vars, bmi_col, after = 5)

tbl1 <- df_full |>
  select(all_of(demo_vars)) |>
  tbl_summary(
    statistic = list(
      all_continuous()  ~ "{mean} ({sd})",
      all_categorical() ~ "{n} ({p}%)"
    ),
    label = list(
      Age_P2                        ~ "Age (years)",
      gender                        ~ "Sex",
      race                          ~ "Race",
      SmokCigNow_P2                 ~ "Current smoker",
      ATS_PackYears_P2              ~ "Pack-years of smoking",
      FEV1pp_P2                     ~ "FEV1 (% predicted)",
      FEV1_FVC_post_P2              ~ "FEV1/FVC",
      lung_density_vnb_P2           ~ "Lung density Phase 2 (HU)",
      meanmal                       ~ "MAL score",
      Change_lung_density_vnb_P2_P3 ~ "Emphysema progression (HU)"
    )
  ) |>
  bold_labels()

tbl1

## Analysis 1 — Differential Protein Expression (Protein ~ MAL)

For each of ~4,979 proteins, fit:
```
log2(protein) ~ meanmal + Age + sex + race + pack-years + current-smoking + BMI + scanner + PC1–PC5
```
Extract the MAL coefficient (β, SE, t, p). Apply Benjamini–Hochberg FDR correction.
Threshold: **FDR < 0.05** = differentially expressed.

In [ ]:
# Build covariate formula string (BMI included if available)
cov_vars <- c("Age_P2", "gender", "race", "ATS_PackYears_P2",
              "SmokCigNow_P2", "scanner_model_clean_P2", paste0("PC", 1:5))
if (!is.null(bmi_col)) cov_vars <- c(cov_vars, bmi_col)
cov_formula <- paste(cov_vars, collapse = " + ")

message("Running protein ~ MAL regression for ", length(protein_cols), " proteins...")
de_results_raw <- run_protein_lm(
  df       = df_full,
  protein_cols = protein_cols,
  exposure = "meanmal",
  covariate_formula = cov_formula
)

# Annotate with gene symbols
de_results <- de_results_raw |>
  mutate(seqID = sub("^log2_X", "", predictor)) |>
  left_join(meta, by = "seqID") |>
  arrange(FDR)

n_sig <- sum(de_results$FDR < 0.05, na.rm = TRUE)
message("Proteins FDR < 0.05: ", n_sig)
print(de_results |> select(gene, target, beta, se, t_stat, p_value, FDR) |> head(20))

## Analysis 2 — GSEA

All proteins are ranked by their **t-statistic** for MAL association and tested against
three gene set collections using `fgsea` (Korotkevich et al. 2021):
- **Hallmark (H):** 50 curated biological process signatures
- **KEGG Legacy (C2:CP:KEGG):** metabolic and signaling pathways
- **Reactome (C2:CP:REACTOME):** detailed mechanistic pathways

In [ ]:
# Rank vector: t-statistic for MAL, named by gene symbol
# Proteins without gene symbol are excluded (unmappable to pathway databases)
ranks_df <- de_results |>
  filter(!is.na(gene) & gene != "") |>
  group_by(gene) |>
  slice_max(abs(t_stat), n = 1, with_ties = FALSE) |>  # one entry per gene
  ungroup()
ranks <- setNames(ranks_df$t_stat, ranks_df$gene)

# Helper: load gene sets and run fgsea
run_gsea <- function(ranks, collection, subcollection = NULL, source_label) {
  gs <- if (is.null(subcollection)) {
    msigdbr(species = "Homo sapiens", collection = collection)
  } else {
    msigdbr(species = "Homo sapiens", collection = collection,
            subcollection = subcollection)
  }
  pathways <- gs |> group_by(gs_name) |>
    summarise(genes = list(gene_symbol), .groups = "drop") |>
    deframe()

  set.seed(42)
  res <- fgsea(pathways = pathways, stats = ranks,
               minSize = 15, maxSize = 500) |>
    as_tibble() |>
    mutate(source = source_label,
           pathway_clean = gsub("^(HALLMARK|KEGG|REACTOME)_", "", pathway),
           pathway_clean = gsub("_", " ", pathway_clean)) |>
    arrange(padj)
  res
}

message("Running GSEA — Hallmark...")
gsea_hallmark <- run_gsea(ranks, "H", NULL, "Hallmark")
message("Running GSEA — KEGG...")
gsea_kegg     <- run_gsea(ranks, "C2", "CP:KEGG_LEGACY", "KEGG")
message("Running GSEA — Reactome...")
gsea_reactome <- run_gsea(ranks, "C2", "CP:REACTOME", "Reactome")

gsea_all <- bind_rows(gsea_hallmark, gsea_kegg, gsea_reactome)

cat(sprintf("Significant pathways (padj<0.05): Hallmark=%d  KEGG=%d  Reactome=%d\n",
  sum(gsea_hallmark$padj < 0.05, na.rm=TRUE),
  sum(gsea_kegg$padj     < 0.05, na.rm=TRUE),
  sum(gsea_reactome$padj < 0.05, na.rm=TRUE)))

print(gsea_all |> filter(padj < 0.05) |>
  select(source, pathway_clean, NES, padj) |> head(20))

## Analysis 3 — STRING Protein–Protein Interaction Network

FDR < 0.05 proteins are mapped to the STRING database (v11.5)
to visualise known and predicted protein–protein interactions.
Score threshold = 400 (medium confidence).

In [ ]:
sig_proteins <- de_results |>
  filter(FDR < 0.05, !is.na(gene), gene != "") |>
  select(gene, beta, FDR)

message("Mapping ", nrow(sig_proteins), " significant proteins to STRING...")

string_db <- STRINGdb$new(version = "11.5", species = 9606,
                          score_threshold = 400, input_directory = OUT_DIR)
mapped    <- string_db$map(as.data.frame(sig_proteins), "gene", removeUnmappedRows = TRUE)
message("Mapped: ", nrow(mapped), " / ", nrow(sig_proteins), " proteins")

# Save network plot
png(file.path(OUT_DIR, "Figure2C_STRING_network.png"), width = 1200, height = 1000, res = 150)
string_db$plot_network(mapped$STRING_id)
dev.off()

# Display inline
string_db$plot_network(mapped$STRING_id)

## Analysis 4 — Adaptive LASSO: MAL Proteomic Risk Score

**Outcome:** `meanmal` (continuous MAL score).  
**Goal:** select a parsimonious set of proteins that predicts MAL,
then collapse them into a single per-subject risk score.

**Procedure:**
1. 70/30 train/test split (`set.seed(2024)`)
2. Ridge regression on residualised MAL → adaptive weights wⱼ = 1/|β̂_ridge_j|
3. Adaptive lasso with 10-fold CV → `lambda.min`
4. Post-selection OLS refit → unbiased β, SE, CI, p
5. MAL_PRS_scaled = (Σⱼ β̂_OLS_j × log₂(protein_ij) − mean) / sd

In [ ]:
# 70/30 split
set.seed(2024)
n_train   <- round(0.70 * nrow(df_full))
train_idx <- sample(nrow(df_full), n_train)
df_train  <- df_full[train_idx, ]
df_test   <- df_full[-train_idx, ]
message("Train: ", nrow(df_train), "  Test: ", nrow(df_test))

# Fixed covariates (unpenalized — known confounders of MAL)
mal_fixed_vars <- c("Age_P2", "gender", "race", "ATS_PackYears_P2",
                    "SmokCigNow_P2", "scanner_model_clean_P2", paste0("PC", 1:5))
if (!is.null(bmi_col)) mal_fixed_vars <- c(mal_fixed_vars, bmi_col)

# Build initial design (uniform weights) for ridge pre-step
d_train_init <- build_design(df_train, protein_cols, mal_fixed_vars)

# Residualise MAL w.r.t. covariates to isolate protein-specific signal
X_cov   <- d_train_init$X[, seq_len(d_train_init$n_fixed), drop = FALSE]
X_prot  <- d_train_init$X[, -seq_len(d_train_init$n_fixed), drop = FALSE]
y_mal   <- df_train$meanmal
y_resid <- residuals(lm.fit(X_cov, y_mal))

# Ridge → adaptive weights
adaptive_weights <- compute_adaptive_weights(X_prot, y_resid)
message("Adaptive weights: min=", round(min(adaptive_weights), 4),
        "  max=", round(max(adaptive_weights), 4))
message("Infinite weights: ", sum(is.infinite(adaptive_weights)))  # must be 0

# Rebuild design matrices with adaptive weights
d_train <- build_design(df_train, protein_cols, mal_fixed_vars, adaptive_weights)
d_test  <- build_design(df_test,  protein_cols, mal_fixed_vars, adaptive_weights,
                        ref_colnames = d_train$colnames)

# Fit adaptive lasso
message("Fitting adaptive lasso for MAL risk score...")
lasso_mal <- fit_adaptive_lasso(d_train$X, y_mal, d_train$pf)

# OLS refit on training set
ols_mal <- refit_ols(d_train$X, y_mal, lasso_mal, "lambda.min",
                     protein_cols, d_train$n_fixed)

# Test set R² (how well does the score predict MAL in held-out data?)
y_pred_test <- as.vector(predict(lasso_mal, newx = d_test$X, s = "lambda.min"))
r2_test     <- cor(df_test$meanmal, y_pred_test)^2
message("Test R² for MAL prediction: ", round(r2_test, 3))

In [ ]:
# Compute MAL proteomic risk score for ALL subjects
prot_coefs <- ols_mal |>
  filter(predictor %in% protein_cols) |>
  mutate(abs_beta = abs(beta)) |>
  left_join(meta |> mutate(predictor = paste0("log2_X", seqID)), by = "predictor")

message("Proteins in MAL risk score: ", nrow(prot_coefs))

# PRS = weighted sum of log2 protein levels
selected_prots <- prot_coefs$predictor
betas_vec      <- setNames(prot_coefs$beta, selected_prots)
X_prs          <- as.matrix(df_full[, selected_prots, drop = FALSE])
prs_raw        <- as.vector(X_prs %*% betas_vec)
MAL_PRS_scaled <- (prs_raw - mean(prs_raw)) / sd(prs_raw)
df_full$MAL_PRS_scaled <- MAL_PRS_scaled

message("MAL PRS: mean=", round(mean(MAL_PRS_scaled), 3),
        "  sd=",  round(sd(MAL_PRS_scaled), 3),
        "  range [", round(min(MAL_PRS_scaled), 2), ", ", round(max(MAL_PRS_scaled), 2), "]")

# Show top proteins
print(prot_coefs |>
  arrange(desc(abs_beta)) |>
  select(gene, target, beta, ci_lower, ci_upper, p_value) |>
  head(20))

## Analysis 5 — Risk Score ~ Emphysema Progression + AIC Comparison

Three nested models compared by AIC (improvement ≥ −2 is meaningful):

| Model | Predictors |
|-------|------------|
| Clinical | Age, sex, race, smoking, pack-years, BMI, scanner |
| PRS only | MAL_PRS_scaled + baseline lung density + scanner |
| Clinical + PRS | All of the above |

In [ ]:
clinical_vars <- paste(c("Age_P2", "gender", "race", "ATS_PackYears_P2",
                          "SmokCigNow_P2", "scanner_model_clean_P2",
                          if (!is.null(bmi_col)) bmi_col else NULL),
                       collapse = " + ")

m_clinical <- lm(as.formula(paste(
  "Change_lung_density_vnb_P2_P3 ~", clinical_vars)), data = df_full)

m_prs <- lm(Change_lung_density_vnb_P2_P3 ~ MAL_PRS_scaled +
              lung_density_vnb_P2 + scanner_model_clean_P2, data = df_full)

m_combined <- lm(as.formula(paste(
  "Change_lung_density_vnb_P2_P3 ~ MAL_PRS_scaled + lung_density_vnb_P2 +",
  clinical_vars)), data = df_full)

# AIC comparison
aic_table <- tibble(
  model    = c("Clinical", "PRS only", "Clinical + PRS"),
  AIC      = c(AIC(m_clinical), AIC(m_prs), AIC(m_combined)),
  delta_vs_clinical = AIC - AIC(m_clinical)
)
cat("AIC comparison (improvement \u2265 \u22122 is meaningful):\n")
print(aic_table)

# PRS coefficient from combined model
prs_summary <- summary(m_combined)$coefficients
prs_ci      <- confint(m_combined)["MAL_PRS_scaled", ]
cat(sprintf("\nMAL PRS \u03b2 = %.3f  95%% CI [%.3f, %.3f]  p = %.2e\n",
  prs_summary["MAL_PRS_scaled", "Estimate"],
  prs_ci[1], prs_ci[2],
  prs_summary["MAL_PRS_scaled", "Pr(>|t|)"]))

## Analysis 6 — Causal Mediation (medflex)

**Model:** MAL → MAL Proteomic Risk Score → Emphysema Progression

Using the natural effects framework (`medflex`; Vansteelandt & Lange 2012):
- **NDE** (Natural Direct Effect): effect of MAL on progression NOT through proteins
- **NIE** (Natural Indirect Effect): effect of MAL on progression THROUGH proteins
- **Total effect** = NDE + NIE
- **Proportion mediated** = NIE / Total

**Supplement:** same mediation for each individual protein in the risk score.

In [ ]:
# Ensure factor variables are properly coded before medflex
df_med <- df_full |>
  mutate(
    gender                = factor(gender),
    race                  = factor(race),
    SmokCigNow_P2         = factor(SmokCigNow_P2),
    scanner_model_clean_P2 = factor(scanner_model_clean_P2)
  )

med_covs <- paste(c("lung_density_vnb_P2", "Age_P2", "gender", "race",
                    "ATS_PackYears_P2", "SmokCigNow_P2", "scanner_model_clean_P2",
                    if (!is.null(bmi_col)) bmi_col else NULL),
                  collapse = " + ")

message("Fitting natural effect model (medflex)...")
imp_data <- neImpute(
  as.formula(paste(
    "Change_lung_density_vnb_P2_P3 ~ meanmal * MAL_PRS_scaled +", med_covs
  )),
  family = gaussian(link = "identity"),
  data   = df_med
)

ne_mod <- neModel(
  as.formula(paste(
    "Change_lung_density_vnb_P2_P3 ~ meanmal0 + meanmal1 +", med_covs
  )),
  family  = gaussian(link = "identity"),
  expData = imp_data,
  se      = "robust"
)

print(summary(ne_mod))

# Extract NDE, NIE, total effect, proportion mediated
ne_coefs   <- summary(ne_mod)$coefficients
nde        <- ne_coefs["meanmal0", "Estimate"]
nie        <- ne_coefs["meanmal1", "Estimate"]
total_eff  <- nde + nie
prop_med   <- nie / total_eff

cat(sprintf("\nNDE  = %.4f  (direct effect of MAL not through proteins)\n", nde))
cat(sprintf("NIE  = %.4f  (indirect effect of MAL through proteins)\n", nie))
cat(sprintf("Total = %.4f\n", total_eff))
cat(sprintf("Proportion mediated = %.1f%%\n", 100 * prop_med))

In [ ]:
# Supplement: mediation for individual proteins in the risk score
# Run only for proteins with p < 0.05 in OLS refit to limit computation
supp_proteins <- prot_coefs |> filter(p_value < 0.05) |> pull(predictor)
message("Running individual mediation for ", length(supp_proteins), " proteins...")

mediation_supp <- map_dfr(supp_proteins, function(prot) {
  tryCatch({
    df_med[["mediator"]] <- df_med[[prot]]
    imp <- neImpute(
      as.formula(paste("Change_lung_density_vnb_P2_P3 ~ meanmal * mediator +", med_covs)),
      family = gaussian(), data = df_med
    )
    mod <- neModel(
      as.formula(paste("Change_lung_density_vnb_P2_P3 ~ meanmal0 + meanmal1 +", med_covs)),
      family = gaussian(), expData = imp, se = "robust"
    )
    coefs <- summary(mod)$coefficients
    tibble(
      predictor = prot,
      NDE       = coefs["meanmal0", "Estimate"],
      NDE_p     = coefs["meanmal0", "Pr(>|z|)"],
      NIE       = coefs["meanmal1", "Estimate"],
      NIE_p     = coefs["meanmal1", "Pr(>|z|)"],
      prop_med  = coefs["meanmal1", "Estimate"] /
                    (coefs["meanmal0", "Estimate"] + coefs["meanmal1", "Estimate"])
    )
  }, error = function(e) NULL)
}) |>
  left_join(meta |> mutate(predictor = paste0("log2_X", seqID)), by = "predictor") |>
  arrange(NIE_p)

print(mediation_supp |> select(gene, target, NDE, NIE, prop_med, NIE_p) |> head(20))

## Figures

In [ ]:
# Figure 2A — Volcano plot: protein ~ MAL
# Colors: Red = up with MAL, Blue = down with MAL, Grey = NS

de_plot <- de_results |>
  mutate(
    significance = case_when(
      FDR < 0.05 & beta > 0 ~ "Up with MAL",
      FDR < 0.05 & beta < 0 ~ "Down with MAL",
      TRUE                  ~ "NS"
    ),
    neg_log10_p = -log10(p_value),
    label = if_else(FDR < 0.05 & rank(FDR) <= 20, gene, NA_character_)
  )

fig2a <- ggplot(de_plot, aes(x = beta, y = neg_log10_p, color = significance)) +
  geom_point(alpha = 0.6, size = 0.8) +
  geom_text_repel(aes(label = label), size = 2.5, max.overlaps = 20, na.rm = TRUE) +
  geom_hline(yintercept = -log10(max(de_results$p_value[de_results$FDR < 0.05],
                                      na.rm = TRUE)),
             linetype = "dashed", color = "grey40") +
  geom_vline(xintercept = 0, linetype = "dotted", color = "grey60") +
  scale_color_manual(values = c("Up with MAL" = "#D6604D",
                                "Down with MAL" = "#4393C3",
                                "NS" = "grey70")) +
  labs(x = expression(beta ~ "(per unit MAL)"),
       y = expression(-log[10](p)),
       color = NULL,
       title = "Differential protein expression with MAL") +
  theme_classic(base_size = 12) +
  theme(legend.position = "top")

ggsave(file.path(OUT_DIR, "Figure2A_volcano.pdf"), fig2a, width=7, height=6)
ggsave(file.path(OUT_DIR, "Figure2A_volcano.png"), fig2a, width=7, height=6, dpi=300)
fig2a

In [ ]:
# Figure 2B — GSEA barplot: top significant pathways across all collections

gsea_sig <- gsea_all |>
  filter(padj < 0.05) |>
  group_by(source) |>
  slice_max(abs(NES), n = 10, with_ties = FALSE) |>
  ungroup() |>
  mutate(
    direction     = if_else(NES > 0, "Enriched in high MAL", "Depleted in high MAL"),
    pathway_label = str_trunc(pathway_clean, 50),
    pathway_label = reorder(pathway_label, NES)
  )

fig2b <- ggplot(gsea_sig, aes(x = NES, y = pathway_label, fill = direction)) +
  geom_col() +
  geom_vline(xintercept = 0, color = "grey30", linewidth = 0.4) +
  scale_fill_manual(values = c("Enriched in high MAL" = "#D6604D",
                               "Depleted in high MAL" = "#4393C3")) +
  facet_wrap(~ source, scales = "free_y", ncol = 1) +
  labs(x = "Normalized Enrichment Score (NES)", y = NULL, fill = NULL,
       title = "GSEA: pathways associated with MAL") +
  theme_classic(base_size = 11) +
  theme(legend.position = "top", strip.background = element_blank(),
        strip.text = element_text(face = "bold"))

ggsave(file.path(OUT_DIR, "Figure2B_GSEA.pdf"), fig2b, width=8, height=10)
ggsave(file.path(OUT_DIR, "Figure2B_GSEA.png"), fig2b, width=8, height=10, dpi=300)
fig2b

In [ ]:
# Figure 3 — Quartile plot: MAL PRS quartile vs effect on emphysema progression

df_full$prs_quartile <- ntile(df_full$MAL_PRS_scaled, 4)

quartile_results <- map_dfr(1:4, function(q) {
  sub_df <- filter(df_full, prs_quartile == q)
  fml    <- as.formula(paste(
    "Change_lung_density_vnb_P2_P3 ~ MAL_PRS_scaled + lung_density_vnb_P2 +",
    clinical_vars
  ))
  mod <- lm(fml, data = sub_df)
  broom::tidy(mod, conf.int = TRUE) |>
    filter(term == "MAL_PRS_scaled") |>
    mutate(quartile = q, n = nrow(sub_df))
})

fig3 <- ggplot(quartile_results,
               aes(x = factor(quartile), y = estimate,
                   ymin = conf.low, ymax = conf.high)) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
  geom_errorbar(width = 0.2, color = "grey40") +
  geom_point(size = 3, color = "#2166AC") +
  labs(
    x     = "Proteomic Risk Score Quartile",
    y     = expression(beta ~ "(HU per 1-SD PRS, 95% CI)"),
    title = "Effect of MAL Proteomic Risk Score on Emphysema Progression by Quartile"
  ) +
  theme_classic(base_size = 12)

ggsave(file.path(OUT_DIR, "Figure3_quartile_plot.pdf"), fig3, width=6, height=5)
ggsave(file.path(OUT_DIR, "Figure3_quartile_plot.png"), fig3, width=6, height=5, dpi=300)
fig3

## Supplementary Figures — Forest Plots

In [ ]:
# Helper: horizontal forest plot with CI bars
forest_plot <- function(df, x_col, xmin_col, xmax_col, label_col,
                        title, xlab, n = 30) {
  top_df <- df |>
    arrange(desc(abs(.data[[x_col]]))) |>
    head(n) |>
    mutate(label = .data[[label_col]],
           label = if_else(is.na(label) | label == "",
                           sub("log2_X", "", predictor), label),
           label = factor(label, levels = rev(label)))

  ggplot(top_df, aes(x = .data[[x_col]], y = label,
                      xmin = .data[[xmin_col]], xmax = .data[[xmax_col]])) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
    geom_errorbarh(height = 0.3, color = "grey40") +
    geom_point(aes(color = .data[[x_col]] > 0), size = 2, show.legend = FALSE) +
    scale_color_manual(values = c("TRUE" = "#D6604D", "FALSE" = "#4393C3")) +
    labs(x = xlab, y = NULL, title = title) +
    theme_classic(base_size = 10)
}

# Supplementary Figure A: top DE proteins ~ MAL
supp_a <- forest_plot(
  df       = de_results |> filter(FDR < 0.05) |>
               mutate(ci_lower = beta - 1.96*se, ci_upper = beta + 1.96*se,
                      predictor = predictor),
  x_col    = "beta", xmin_col = "ci_lower", xmax_col = "ci_upper",
  label_col = "gene",
  title    = "Top differentially expressed proteins — association with MAL",
  xlab     = expression(beta ~ "(HU per log"[2] ~ "unit protein, MAL association)")
)
ggsave(file.path(OUT_DIR, "SuppFig_forest_MAL.pdf"), supp_a, width=8, height=10)
ggsave(file.path(OUT_DIR, "SuppFig_forest_MAL.png"), supp_a, width=8, height=10, dpi=300)
supp_a

In [ ]:
# Supplementary Figure B: DE proteins ~ emphysema progression
# Run lm(progression ~ protein + covariates) for the significant DE proteins
sig_prot_cols <- de_results |> filter(FDR < 0.05) |> pull(predictor)

prog_results <- run_protein_lm(
  df       = df_full,
  protein_cols = sig_prot_cols,
  exposure = "Change_lung_density_vnb_P2_P3",
  covariate_formula = paste(c("lung_density_vnb_P2", cov_vars), collapse = " + ")
) |>
  mutate(seqID = sub("^log2_X", "", predictor)) |>
  left_join(meta, by = "seqID") |>
  mutate(ci_lower = beta - 1.96*se, ci_upper = beta + 1.96*se)

supp_b <- forest_plot(
  df        = prog_results,
  x_col     = "beta", xmin_col = "ci_lower", xmax_col = "ci_upper",
  label_col = "gene",
  title     = "Top differentially expressed proteins — association with emphysema progression",
  xlab      = expression(beta ~ "(HU per log"[2] ~ "unit protein, progression association)")
)
ggsave(file.path(OUT_DIR, "SuppFig_forest_progression.pdf"), supp_b, width=8, height=10)
ggsave(file.path(OUT_DIR, "SuppFig_forest_progression.png"), supp_b, width=8, height=10, dpi=300)
supp_b

## Save Outputs

| File | Contents |
|------|----------|
| `mal_de_results.csv` | All protein ~ MAL associations (β, SE, t, p, FDR, gene) |
| `mal_gsea_results.csv` | GSEA results across Hallmark, KEGG, Reactome |
| `mal_risk_score_weights.csv` | Selected proteins + OLS β, SE, CI, p |
| `mal_risk_score_subjects.csv` | Per-subject MAL_PRS_scaled + outcome + split |
| `mal_mediation_results.csv` | medflex NDE/NIE/proportion mediated (full score) |
| `mal_mediation_supp.csv` | Individual protein mediation results |

In [ ]:
write_csv(de_results |> select(gene, target, full_name, beta, se, t_stat, p_value, FDR),
          file.path(OUT_DIR, "mal_de_results.csv"))
message("Saved: mal_de_results.csv")

write_csv(gsea_all |> select(source, pathway, pathway_clean, NES, padj, size),
          file.path(OUT_DIR, "mal_gsea_results.csv"))
message("Saved: mal_gsea_results.csv")

write_csv(prot_coefs |> select(gene, target, beta, se, ci_lower, ci_upper, p_value),
          file.path(OUT_DIR, "mal_risk_score_weights.csv"))
message("Saved: mal_risk_score_weights.csv")

prs_subject_df <- tibble(
  sid            = df_full$sid,
  MAL_PRS_scaled = df_full$MAL_PRS_scaled,
  meanmal        = df_full$meanmal,
  Change_lung_density_vnb_P2_P3 = df_full$Change_lung_density_vnb_P2_P3,
  split = case_when(
    df_full$sid %in% df_train$sid ~ "train",
    TRUE                          ~ "test"
  )
)
write_csv(prs_subject_df, file.path(OUT_DIR, "mal_risk_score_subjects.csv"))
message("Saved: mal_risk_score_subjects.csv")

# Mediation results (full risk score)
ne_coefs_df <- as_tibble(summary(ne_mod)$coefficients, rownames = "term") |>
  mutate(effect = recode(term,
    meanmal0 = "NDE (direct)",
    meanmal1 = "NIE (indirect via proteins)"
  ))
write_csv(ne_coefs_df, file.path(OUT_DIR, "mal_mediation_results.csv"))
message("Saved: mal_mediation_results.csv")

if (nrow(mediation_supp) > 0) {
  write_csv(mediation_supp |> select(gene, target, NDE, NDE_p, NIE, NIE_p, prop_med),
            file.path(OUT_DIR, "mal_mediation_supp.csv"))
  message("Saved: mal_mediation_supp.csv")
}

message("=== Analysis complete ===")